In [ ]:
import random
import matplotlib as mplt
import matplotlib.pyplot as plt
from boltons.setutils import IndexedSet
import numpy as np
import itertools

In [ ]:
def get_boundaries(square,diagonally=False):
    up=(square[0],square[1]-1)
    down=(square[0],square[1]+1)
    left=(square[0]-1,square[1])
    right=(square[0]+1,square[1])
    return [up, down, left,right]

def generate_island(n):
    "Generates an island formed by n joint squares"
    "could easily add diagonals to boundary as well"
    if n<1:
        return []
    #island=[(0,0)] #for debugging, but gives order
    centroid=(0,0)
    island=set() #for performance, but I do not know how big are the islands
    island.add(centroid)
    boundaries=IndexedSet(get_boundaries(centroid)) #needs to be indexable for randomly picking elements, but also a set to avoid duplicates
    
    for _ in range(n-1):
        #generate a new square from the possibilities (boundaries)
        new_square=random.choice(boundaries)
        
        #anex it to island
        #island.append(new_square) #for debugging
        island.add(new_square)
        
        #now it is not a boundary and cannot be generated again, so remove it
        boundaries.remove(new_square)
        
        #update boundaries
        for boundary in get_boundaries(new_square):
            if boundary not in island:
                boundaries.add(boundary)
        
    return list(island)

#From https://stackoverflow.com/questions/43971138/python-plotting-colored-grid-based-on-values
def plot_colored_grid(data, colors=['white', 'black',"green"], bounds=[0, 0.5, 1.5, 2], grid=True, labels=True, frame=True):
    """Plot 2d matrix with grid with well-defined colors for specific boundary values.

    :param data: 2d matrix
    :param colors: colors
    :param bounds: bounds between which the respective color will be plotted
    :param grid: whether grid should be plotted
    :param labels: whether labels should be plotted
    :param frame: whether frame should be plotted
    """

    # create discrete colormap
    cmap = mplt.colors.ListedColormap(colors)
    norm = mplt.colors.BoundaryNorm(bounds, cmap.N)

    # enable or disable frame
    plt.figure(frameon=frame)
    plt.gca().invert_yaxis()

    # show grid
    if grid:
        #plt.grid(axis='both', color='k', linewidth=2) 
        plt.xticks(np.arange(0, data.shape[1], 1))  # correct grid sizes
        plt.yticks(np.arange(0, data.shape[0], 1))

    # disable labels
    if not labels:
        plt.tick_params(bottom=False, top=False, left=False, right=False, labelbottom=False, labelleft=False) 
    # plot data matrix
    plt.imshow(data, cmap=cmap, norm=norm)

    # display main axis 
    plt.show()

def format_island(island_list,leeway=False):
    #x and y are swapped into col row notation later
    #make 0,0 top left instead of centroid
    start=island_list[0]
    minx=start[0]
    maxx=start[0]
    miny=start[1]
    maxy=start[1]
    
    for square in island_list[1:]:
        minx=min(minx,square[0])
        maxx=max(maxx,square[0])
        miny=min(miny,square[1])
        maxy=max(maxy,square[1])
    
    if leeway:
        #to leave space around the island for the max flow algorithm
        #this way the indexing is consistent for both, and easier to visualize
        minx-=1
        miny-=1
        maxx+=1
        maxy+=1
    
    # to numpy 2D array
    I=np.zeros((maxx-minx+1,maxy-miny+1),dtype=np.int8)
    newisland_list=[]
    
    for square in island_list:
        I[square[0]-minx][square[1]-miny]=1
        newisland_list.append((square[0]-minx,square[1]-miny))
        
    return I,newisland_list


In [ ]:
random.seed(0)
N=50
island,island_list=format_island(generate_island(N))
island_list=sorted(island_list)
print(island_list)
print(island)
plot_colored_grid(island)

In [ ]:
def get_adjacent(cell):
    x=cell[0]
    y=cell[1]
    return [(x,y-1),(x,y+1),(x-1,y),(x+1,y)]

def get_distance_dict(island,island_list):
    rows,cols=island.shape
    distance_dict={}
    # Calculate manhattan distance from each "start_cell" to any other cell
    for start_cell in island_list:
        distances_to=np.full(island.shape, -1, dtype=int)
        distances_to[*start_cell]=0
        distance=0
        at_prev_distance=[start_cell]
        while True:
            added_any=False
            distance+=1
            at_currrent_distance=[]
            for apd in at_prev_distance:
                for neighbour in get_adjacent(apd):
                    # if neighbour on the island
                    if 0<=neighbour[0]<rows and 0<=neighbour[1]<cols and island[*neighbour]!=0:
                        # if neighbour not visited previously
                        if distances_to[*neighbour]==-1:
                            # then add neighbour to currently visited and set new distance
                            added_any=True
                            at_currrent_distance.append(neighbour)
                            distances_to[*neighbour]=distance
                at_prev_distance=at_currrent_distance
            if not added_any:
                break
        distance_dict[start_cell]=distances_to
        #print(distances_to)
    return distance_dict

def brute_force(island,island_list,nof_start_grass):
    distance_dict=get_distance_dict(island,island_list)
    solutions_for_each_time={}
    island_set=set(island_list)
    for choices in itertools.combinations(island_list,nof_start_grass):
        choices_set=set(choices)
        targets=island_set.difference(choices_set)
        distances_to_target=[]
        for target in targets:
            distances_to_target.append(min([distance_dict[choice][*target] for choice in choices]))
            # print(target)
            # print(int(distances_to_target[-1]))
        time=max(distances_to_target)
        #print(time)
        if time not in solutions_for_each_time:
            solutions_for_each_time[time]=[]
        solutions_for_each_time[time].append(choices)
    sorted_solutions=sorted(solutions_for_each_time.items(), key=lambda item: item[0])
    #print(sorted_solutions)
    steps=sorted_solutions[0][0]
    print("Steps", steps)
    print("Number of best solutions", len(sorted_solutions[0][1]))
    for solution in sorted_solutions[0][1]:
        new_island=island.copy()
        for choice in solution:
            new_island[*choice]=2
        print(solution)
        
        plot_colored_grid(new_island)
        
brute_force(island,island_list,2)